<a href="https://colab.research.google.com/github/iavorskii777/Zashita_Informatsii_Praktikum/blob/my-first-site/tema4_zashita_informatsii_praktikum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Тема 4. Защита информации в информационно-телекоммуникационных сетях

**Практикум на Python для Google Colaboratory.**  
Автор-исполнитель: Василий Яворский  
Дата: 06.11.2026

## Цели практикума
- Рассмотреть задачи защиты информации в ИТКС.
- Смоделировать уязвимости линий передачи и типовые сетевые атаки.
- Разобрать предпосылки компьютерных инцидентов и основы их расследования.
- Практически применить межсетевой экран, контроль доступа, антивирусную эвристику.
- Освоить симметричное и асимметричное шифрование, электронную подпись.

> **Рекомендация:** запускайте ячейки последовательно. Для раздела с электронной подписью при необходимости установите библиотеку `cryptography`.

## 1. Сущность проблемы защиты информации в ИТКС

Информационно-телекоммуникационная сеть (ИТКС) — совокупность каналов связи, узлов коммутации, серверов, конечных устройств и программного обеспечения, обеспечивающих передачу, обработку и хранение данных. Защита информации направлена на обеспечение свойств **конфиденциальности**, **целостности** и **доступности** (модель CIA).

**Основные задачи:**
- предотвращение несанкционированного доступа;
- контроль целостности программ и данных;
- обеспечение доступности сервисов;
- регистрация событий и расследование инцидентов;
- криптографическая защита передаваемых данных.

**Практическая модель CIA:** ниже реализуем простую проверку политики доступа.

In [ ]:
# Практическая модель разграничения доступа (Confidentiality, Integrity, Availability)
USERS = {
    'admin': {'role': 'administrator', ' clearance': 3},
    'analyst': {'role': 'analyst', ' clearance': 2},
    'guest': {'role': 'guest', ' clearance': 1}
}

RESOURCES = {
    'public_report': 1,
    'financial_data': 2,
    'system_configuration': 3
}

def check_access(user, resource):
    if user not in USERS:
        return 'DENY: неизвестный пользователь'
    clearance = USERS[user].get(' clearance', 0)
    required = RESOURCES[resource]
    return 'ALLOW' if clearance >= required else 'DENY: недостаточный уровень допуска'

for user in ['admin', 'analyst', 'guest', 'intruder']:
    for resource in RESOURCES:
        print(f'{user:10s} -> {resource:22s}: {check_access(user, resource)}')

admin      -> public_report         : ALLOW
admin      -> financial_data        : ALLOW
admin      -> system_configuration  : ALLOW
analyst    -> public_report         : ALLOW
analyst    -> financial_data        : ALLOW
analyst    -> system_configuration  : DENY: недостаточный уровень допуска
guest      -> public_report         : ALLOW
guest      -> financial_data        : DENY: недостаточный уровень допуска
guest      -> system_configuration  : DENY: недостаточный уровень допуска
intruder   -> public_report         : DENY: неизвестный пользователь
intruder   -> financial_data        : DENY: неизвестный пользователь
intruder   -> system_configuration  : DENY: неизвестный пользователь


**Вывод:** даже простая матрица «пользователь — ресурс — уровень допуска» позволяет формализовать политику конфиденциальности. В реальных системах ее дополняют аутентификацией, журналированием, шифрованием и контролем целостности.

**Примеры для изучения:**
- GitHub: [OWASP Cheat Sheet Series](https://github.com/OWASP/CheatSheetSeries)
- Kaggle: [Cybersecurity datasets](https://www.kaggle.com/datasets?search=cybersecurity)
- Хабр: [Информационная безопасность](https://habr.com/ru/hubs/infosecurity/)

## 2. Уязвимость линий передачи информации

Каналы передачи уязвимы к перехвату, изменению, подмене источника и нарушению доступности. Продемонстрируем три базовых свойства защищенного канала:

1. **Конфиденциальность** — данные недоступны третьей стороне.
2. **Целостность** — изменение данных обнаруживается.
3. **Аутентичность** — подтверждается отправитель.

Модель перехвата: передаваемое сообщение наблюдается «злоумышленником посередине».

In [ ]:
# Модель канала связи: отправитель -> канал -> получатель
# Злоумышленник может перехватывать (E), изменять (M) или блокировать (D) трафик

def transmit(message, attacker='none'):
    if attacker == 'eavesdropper':
        print('[Атака] Перехват сообщения:', message)
        return message
    if attacker == 'modifier':
        tampered = message.replace('Оплата', 'Отмена')
        print('[Атака] Сообщение изменено:', tampered)
        return tampered
    if attacker == 'denial':
        print('[Атака] Сообщение заблокировано (DoS)')
        return None
    return message

msg = 'Оплата счета №128: 50 000 руб.'
for attack in ['none', 'eavesdropper', 'modifier', 'denial']:
    print('---')
    received = transmit(msg, attack)
    print('Получено:', received)

---
Получено: Оплата счета №128: 50 000 руб.
---
[Атака] Перехват сообщения: Оплата счета №128: 50 000 руб.
Получено: Оплата счета №128: 50 000 руб.
---
[Атака] Сообщение изменено: Отмена счета №128: 50 000 руб.
Получено: Отмена счета №128: 50 000 руб.
---
[Атака] Сообщение заблокировано (DoS)
Получено: None


In [ ]:
# Контроль целостности: хеш-функция SHA-256
import hashlib

def digest(data: str) -> str:
    return hashlib.sha256(data.encode('utf-8')).hexdigest()

original = 'Сумма перевода: 50 000 руб.'
original_hash = digest(original)
modified = 'Сумма перевода: 5 000 руб.'
modified_hash = digest(modified)

print('Хеш исходного сообщения: ', original_hash)
print('Хеш измененного сообщения:', modified_hash)
print('Целостность нарушена:', original_hash != modified_hash)

Хеш исходного сообщения:  a336f68a24c34e7f2f86232a481bd1cfd702fad59a5999ac2aff79c06875dea0
Хеш измененного сообщения: 2f2aaebe451f9f16aadc6190867fc1ce04027c0330a260a3d5ea087a8f43ae92
Целостность нарушена: True


**Вывод:** криптографическая хеш-функция выявляет даже минимальное изменение сообщения. Однако хеш сам по себе не защищает от подмены и хеша, и сообщения: для этого нужна электронная подпись или код аутентификации сообщения (MAC).

**Ресурсы:** [NIST FIPS 180-4: Secure Hash Standard](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), [Hugging Face — security models](https://huggingface.co/models?search=cybersecurity).

## 3. Компьютерные преступления и инциденты

Компьютерное преступление — противоправное деяние, в котором компьютер, сеть либо информация выступают предметом, орудием или средством совершения преступления. В УК РФ к ним относятся, в частности:

- неправомерный доступ к охраняемой законом компьютерной информации (ст. 272 УК РФ);
- создание, использование и распространение вредоносных программ (ст. 273 УК РФ);
- нарушение правил эксплуатации средств хранения, обработки или передачи информации (ст. 274 УК РФ);
- неправомерное воздействие на критическую информационную инфраструктуру РФ (ст. 274.1 УК РФ).

**Особенности расследования:** цифровая природа доказательств, необходимость фиксации метаданных, журналов, сетевого трафика и образов носителей; важность цепочки хранения доказательств (chain of custody).

**Предпосылки компьютерных инцидентов:** ошибки конфигурации, слабые пароли, отсутствие обновлений, фишинг, инсайдеры, уязвимое ПО, недостаточный контроль привилегий.

In [ ]:
# Простая система регистрации и первичного анализа компьютерных инцидентов
from datetime import datetime

class Incident:
    def __init__(self, incident_id, source_ip, target, event_type, severity):
        self.incident_id = incident_id
        self.timestamp = datetime.now().isoformat(timespec='seconds')
        self.source_ip = source_ip
        self.target = target
        self.event_type = event_type
        self.severity = severity

    def to_dict(self):
        return self.__dict__

def triage(incident):
    rules = {
        'brute_force': 'Проверить журналы аутентификации, временно заблокировать IP',
        'malware': 'Изолировать хост, снять образ памяти и диска, запустить антивирус',
        'phishing': 'Заблокировать ссылку/вложение, уведомить пользователей, проверить почтовые правила',
        'data_exfiltration': 'Ограничить исходящий трафик, провести форензику, уведомить ответственных'
    }
    return rules.get(incident.event_type, 'Провести стандартное расследование')

incidents = [
    Incident(1, '203.0.113.45', 'vpn.company.ru', 'brute_force', 'high'),
    Incident(2, '10.0.2.15', 'workstation-07', 'malware', 'critical'),
    Incident(3, '198.51.100.9', 'mail.company.ru', 'phishing', 'medium')
]

for inc in incidents:
    print(inc.to_dict())
    print('Рекомендация:', triage(inc), '\n')

{'incident_id': 1, 'timestamp': '2026-11-06T12:18:54', 'source_ip': '203.0.113.45', 'target': 'vpn.company.ru', 'event_type': 'brute_force', 'severity': 'high'}
Рекомендация: Проверить журналы аутентификации, временно заблокировать IP 

{'incident_id': 2, 'timestamp': '2026-11-06T12:18:54', 'source_ip': '10.0.2.15', 'target': 'workstation-07', 'event_type': 'malware', 'severity': 'critical'}
Рекомендация: Изолировать хост, снять образ памяти и диска, запустить антивирус 

{'incident_id': 3, 'timestamp': '2026-11-06T12:18:54', 'source_ip': '198.51.100.9', 'target': 'mail.company.ru', 'event_type': 'phishing', 'severity': 'medium'}
Рекомендация: Заблокировать ссылку/вложение, уведомить пользователей, проверить почтовые правила 



**Вывод:** система реагирования на инциденты должна не только фиксировать событие, но и предлагать проверяемые действия: изоляцию узла, сохранение доказательств, блокировку индикаторов компрометации и уведомление владельцев активов.

**Полезные материалы:** [ГОСТ Р 59709-2022 / управление инцидентами — поиск в КонсультантПлюс](https://www.consultant.ru/), [The DFIR Report](https://thedfirreport.com/), [MITRE ATT&CK](https://attack.mitre.org/).

## 4. Межсетевой экран

Межсетевой экран (МЭ, firewall) контролирует сетевой трафик по правилам: IP-адресам, портам, протоколам, направлениям соединений и содержимому пакетов. Различают пакетные фильтры, stateful-МЭ и МЭ уровня приложений (WAF).

Реализуем упрощенный stateful-фильтр с белым списком и журналированием.

In [ ]:
# Упрощенный межсетевой экран
import ipaddress

class SimpleFirewall:
    def __init__(self):
        self.rules = []
        self.log = []

    def add_rule(self, action, src, dst_port, protocol):
        self.rules.append({'action': action, 'src': src, 'dst_port': dst_port, 'protocol': protocol})

    def inspect(self, src_ip, dst_port, protocol):
        decision = 'DENY'
        for rule in self.rules:
            src_match = (rule['src'] == 'any') or ipaddress.ip_address(src_ip) in ipaddress.ip_network(rule['src'])
            if src_match and rule['dst_port'] == dst_port and rule['protocol'] == protocol:
                decision = rule['action'].upper()
                break
        self.log.append({
            'time': datetime.now().isoformat(timespec='seconds'),
            'src': src_ip, 'dst_port': dst_port,
            'protocol': protocol, 'decision': decision
        })
        return decision

fw = SimpleFirewall()
fw.add_rule('allow', '192.168.1.0/24', 443, 'TCP')
fw.add_rule('allow', 'any', 53, 'UDP')
fw.add_rule('deny', 'any', 23, 'TCP')

traffic = [
    ('192.168.1.10', 443, 'TCP'),
    ('192.168.1.10', 23, 'TCP'),
    ('203.0.113.77', 443, 'TCP'),
    ('192.168.1.15', 53, 'UDP')
]

for packet in traffic:
    decision = fw.inspect(*packet)
    print(packet, '->', decision)

print('\nЖурнал МЭ:')
for row in fw.log:
    print(row)

('192.168.1.10', 443, 'TCP') -> ALLOW
('192.168.1.10', 23, 'TCP') -> DENY
('203.0.113.77', 443, 'TCP') -> DENY
('192.168.1.15', 53, 'UDP') -> ALLOW

Журнал МЭ:
{'time': '2026-11-06T12:18:57', 'src': '192.168.1.10', 'dst_port': 443, 'protocol': 'TCP', 'decision': 'ALLOW'}
{'time': '2026-11-06T12:18:57', 'src': '192.168.1.10', 'dst_port': 23, 'protocol': 'TCP', 'decision': 'DENY'}
{'time': '2026-11-06T12:18:57', 'src': '203.0.113.77', 'dst_port': 443, 'protocol': 'TCP', 'decision': 'DENY'}
{'time': '2026-11-06T12:18:57', 'src': '192.168.1.15', 'dst_port': 53, 'protocol': 'UDP', 'decision': 'ALLOW'}


**Вывод:** МЭ снижает поверхность атаки, но не заменяет обновление ПО, антивирусную защиту и контроль учетных записей. Принцип наименьших привилегий предполагает правило «запрещено все, что явно не разрешено».

**Примеры:** [pfSense на GitHub](https://github.com/pfsense/pfsense), [OPNsense на GitHub](https://github.com/opnsense/core), [Kaggle: Network Intrusion Detection](https://www.kaggle.com/datasets?search=network+intrusion+detection).

## 5. Защита ПО и данных ПЭВМ

Основные методы:
- разграничение доступа и принцип наименьших привилегий;
- контроль целостности файлов и конфигураций;
- резервное копирование по правилу 3-2-1;
- обновление программного обеспечения;
- шифрование дисков и резервных копий;
- журналирование действий пользователей.

Ниже — контроль целостности критичных файлов и простая система обнаружения подозрительных изменений.

In [ ]:
# Мониторинг целостности файлов
import os, json, hashlib

baseline = {}
files = {
    'system_config.txt': 'allow_remote_access=false\nlogging=enabled',
    'payment_app.py': 'def pay(amount):\n    return amount * 1.0'
}

for name, content in files.items():
    baseline[name] = hashlib.sha256(content.encode()).hexdigest()

# Имитация несанкционированного изменения конфигурации
files['system_config.txt'] = 'allow_remote_access=true\nlogging=disabled'

for name, content in files.items():
    current = hashlib.sha256(content.encode()).hexdigest()
    status = 'OK' if current == baseline[name] else 'ALERT: файл изменен'
    print(f'{name:20s} {status}')

system_config.txt    ALERT: файл изменен
payment_app.py       OK


In [ ]:
# Резервное копирование 3-2-1 и проверка пароля
import re, getpass

def password_strength(password):
    checks = {
        'длина >= 12': len(password) >= 12,
        'есть строчные буквы': bool(re.search(r'[a-zа-я]', password)),
        'есть заглавные буквы': bool(re.search(r'[A-ZА-Я]', password)),
        'есть цифры': bool(re.search(r'\d', password)),
        'есть специальные символы': bool(re.search(r'[^A-Za-zА-Яа-я0-9]', password))
    }
    score = sum(checks.values())
    return score, checks

for pwd in ['123456', 'Password1', 'Qw!9#zLm42Vt']:
    score, checks = password_strength(pwd)
    print(pwd, '->', score, '/5', checks)

123456 -> 1 /5 {'длина >= 12': False, 'есть строчные буквы': False, 'есть заглавные буквы': False, 'есть цифры': True, 'есть специальные символы': False}
Password1 -> 3 /5 {'длина >= 12': False, 'есть строчные буквы': True, 'есть заглавные буквы': True, 'есть цифры': True, 'есть специальные символы': False}
Qw!9#zLm42Vt -> 5 /5 {'длина >= 12': True, 'есть строчные буквы': True, 'есть заглавные буквы': True, 'есть цифры': True, 'есть специальные символы': True}


**Вывод:** защита данных ПЭВМ строится по принципу эшелонированной обороны: ни одно средство не является достаточным само по себе. Контроль целостности выявляет компрометацию, а стойкие пароли и резервные копии уменьшают последствия инцидента.

**Ресурсы:** [CIS Controls](https://www.cisecurity.org/controls), [NIST Cybersecurity Framework](https://www.nist.gov/cyberframework).

## 6. Криптографические методы защиты

**Симметричное шифрование** использует один секретный ключ для шифрования и расшифрования. Оно быстрое, но требует защищенной доставки ключа.

**Асимметричное шифрование** использует пару ключей: открытый — для шифрования или проверки подписи, закрытый — для расшифрования или создания подписи.

**Электронная подпись** обеспечивает целостность, аутентичность и неотказуемость: подпись создается закрытым ключом, проверяется открытым ключом.

In [ ]:
# Симметричное шифрование Fernet (AES-128-CBC + HMAC)
# Если библиотека не установлена: !pip install cryptography
from cryptography.fernet import Fernet

key = Fernet.generate_key()
cipher = Fernet(key)
message = 'Конфиденциальные данные: договор №45 от 06.11.2026'
token = cipher.encrypt(message.encode())
decrypted = cipher.decrypt(token).decode()

print('Ключ (хранить в защищенном хранилище):', key.decode())
print('Шифртекст:', token.decode())
print('Расшифровано:', decrypted)

# Попытка расшифрования другим ключом
wrong_cipher = Fernet(Fernet.generate_key())
try:
    wrong_cipher.decrypt(token)
except Exception as e:
    print('Расшифрование чужим ключом отклонено:', type(e).__name__)

Ключ (хранить в защищенном хранилище): PE2Hr5K0DC1l-SKn3TZfNXRlQ0QRQ-HSKHKpYjwwCYc=
Шифртекст: gAAAAABq7cW6u7YAvIOsHP44xM2shxAP5naaqCYfj_ihAicAirrjqwC6__RMHXH3Q5CBnCNrQNZFy9GbAogE0BGofHDXNTxR-SXEFrnRnZilcDNFgCtCjyxBizv7WqgtMTm8MOg5Qt8qyH7rzUyIdQ0cWj95HJQGHv6pVi8WZNYDTdaRJ95KPKS_q7H1aHkmNYAHCWgkbLxF
Расшифровано: Конфиденциальные данные: договор №45 от 06.11.2026
Расшифрование чужим ключом отклонено: InvalidToken


**Вывод:** на практике применяют гибридную схему: асимметричная криптография защищает обмен сеансовым ключом, а симметричное шифрование обеспечивает быструю защиту большого объема данных. Электронная подпись связывает документ, время подписания и владельца закрытого ключа.

**Ресурсы:** [pyca/cryptography на GitHub](https://github.com/pyca/cryptography), [Kaggle: cryptography](https://www.kaggle.com/datasets?search=cryptography), [Hugging Face: security](https://huggingface.co/models?search=security).

## 7. Безопасность в Интернете: фишинг и вредоносное ПО

Типовые угрозы Интернета: фишинг, вредоносные вложения, вредоносные ссылки, подмена DNS, межсайтовый скриптинг, утечки данных. Компьютерные вирусы — программы, способные к самовоспроизведению и выполнению вредоносных действий; современные вредоносные программы также включают трояны, шпионское ПО, программы-вымогатели и ботнеты.

Антивирусные продукты используют сигнатуры, эвристический анализ, поведенческое обнаружение, песочницу и облачную репутацию файлов.

In [ ]:
# Эвристический детектор подозрительных признаков в письме/файле
import re

SUSPICIOUS_PATTERNS = {
    'срочность': r'(срочно|немедленно|в течение 24 часов|account suspended)',
    'запрос учетных данных': r'(введите пароль|подтвердите логин|credentials|password)',
    'сокращатель ссылок': r'(bit\.ly|tinyurl\.com|t\.me/)',
    'исполняемое вложение': r'\.(exe|scr|bat|cmd|js|vbs)$',
    'макрос в документе': r'\.(docm|xlsm|pptm)$'
}

def scan(text):
    findings = []
    for name, pattern in SUSPICIOUS_PATTERNS.items():
        if re.search(pattern, text, flags=re.IGNORECASE):
            findings.append(name)
    risk = 'Высокий' if len(findings) >= 2 else 'Средний' if findings else 'Низкий'
    return risk, findings

samples = [
    'Ваш аккаунт заблокирован. Срочно введите пароль: http://bit.ly/login',
    'Отчет за октябрь.docx',
    'invoice_2026.pdf.exe',
    'Презентация проекта.pptm'
]

for s in samples:
    risk, findings = scan(s)
    print(f'{s:65s} -> {risk}: {findings}')

Ваш аккаунт заблокирован. Срочно введите пароль: http://bit.ly/login -> Высокий: ['срочность', 'запрос учетных данных', 'сокращатель ссылок']
Отчет за октябрь.docx                                             -> Низкий: []
invoice_2026.pdf.exe                                              -> Средний: ['исполняемое вложение']
Презентация проекта.pptm                                          -> Средний: ['макрос в документе']


In [ ]:
# Упрощенная сигнатурная проверка: известные хеши вредоносных файлов
import hashlib

malware_hashes = {
    'e99a18c428cb38d5f260853678922e03abd8334e8f9d0f3d0b8c1a2f6b9d4e7f': 'Trojan.Generic.Demo'
}

def file_hash(data: bytes):
    return hashlib.sha256(data).hexdigest()

file_samples = {
    'report.pdf': b'%PDF-1.7 normal document',
    'update.exe': b'MZ fake executable',
    'known_malware.bin': b'known malicious payload'
}

# Добавим заранее вычисленный хеш для демонстрации срабатывания сигнатуры
known_hash = file_hash(b'known malicious payload')
malware_hashes[known_hash] = 'Trojan.Generic.Demo'

for name, data in file_samples.items():
    h = file_hash(data)
    verdict = malware_hashes.get(h, 'Чисто / не найдено в базе')
    print(f'{name:20s} {h[:16]}... -> {verdict}')

report.pdf           0fec3cdf0b84330b... -> Чисто / не найдено в базе
update.exe           8ac8dd84944893b4... -> Чисто / не найдено в базе
known_malware.bin    20e230de2bf8b93f... -> Trojan.Generic.Demo


**Вывод:** сигнатурный анализ надежен для известных угроз, но не обнаруживает новые образцы. Поэтому современная защита должна сочетать эвристику, поведенческий анализ, песочницу, обновление баз и обучение пользователей.

**Примеры:** [VirusTotal](https://www.virustotal.com/), [ClamAV на GitHub](https://github.com/Cisco-Talos/clamav), [Kaggle: malware datasets](https://www.kaggle.com/datasets?search=malware), [Hugging Face: malware detection](https://huggingface.co/models?search=malware+detection).

## 8. Итоговые выводы

1. Защита информации в ИТКС — это непрерывный процесс управления рисками, а не однократная установка программных средств.
2. Уязвимость каналов связи требует комплексной защиты: шифрования, контроля целостности, аутентификации и мониторинга.
3. Расследование компьютерных преступлений опирается на цифровые доказательства, журналы, сетевые артефакты и соблюдение процессуальных требований.
4. Межсетевой экран реализует политику сетевого доступа, но должен дополняться обновлениями, антивирусом, контролем учетных записей и обучением сотрудников.
5. Криптография решает задачи конфиденциальности, целостности, аутентичности и неотказуемости; симметричные и асимметричные методы применяются совместно.
6. Эффективная защита от вредоносного ПО основана на нескольких независимых уровнях обнаружения и реагирования.

## Библиографический список
1. Уголовный кодекс Российской Федерации, ст. 272–274.1. — [КонсультантПлюс](https://www.consultant.ru/document/cons_doc_LAW_10699/).
2. ГОСТ Р 50922-2006. Защита информации. Основные термины и определения. — [Docs.cntd.ru](https://docs.cntd.ru/document/1200038310).
3. NIST. Cybersecurity Framework 2.0. — [nist.gov](https://www.nist.gov/cyberframework).
4. OWASP. Cheat Sheet Series. — [GitHub](https://github.com/OWASP/CheatSheetSeries).
5. MITRE. ATT&CK Framework. — [attack.mitre.org](https://attack.mitre.org/).
6. Python Cryptographic Authority. `cryptography` Documentation. — [cryptography.io](https://cryptography.io/).
7. Cisco Talos. ClamAV. — [GitHub](https://github.com/Cisco-Talos/clamav).
8. The DFIR Report. Digital Forensics and Incident Response. — [thedfirreport.com](https://thedfirreport.com/).
9. Kaggle. Cybersecurity and Malware Datasets. — [kaggle.com](https://www.kaggle.com/datasets).
10. Хабр. Хаб «Информационная безопасность». — [habr.com](https://habr.com/ru/hubs/infosecurity/).